[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/openai/openai_agents_evaluator_optimizer.ipynb)


# <center>OpenAI 智能体模式：评估与优化智能体</center>

本入门指南介绍如何使用 `openai-agents` 库，根据 LLM 反馈迭代生成输出。

LLM 的初次生成往往不够理想。可以通过智能体循环让 LLM 提供反馈，再据此改进输出。

下面使用这一模式构建财务报告系统：
1. **报告智能体（生成）**：为指定股票代码生成报告。
2. **评估智能体（反馈）**：评估报告，并提出改进建议。

### 安装库

In [ ]:
# 安装 OpenAI 基础库
!pip install -q openai openai-agents pydantic

# 安装 OpenInference/OpenTelemetry 追踪的可选库
!pip install -q arize-phoenix-otel openinference-instrumentation-openai-agents openinference-instrumentation-openai

### 设置密钥
将 OpenAI API 密钥添加到环境变量 `OPENAI_API_KEY`。

从 [app.phoenix.arize.com](https://app.phoenix.arize.com) 的设置页面复制 Phoenix `API_KEY`。

In [ ]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("🔑 请输入 OpenAI API 密钥： ")

if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

### 设置追踪

In [ ]:
from phoenix.otel import register

tracer_provider = register(
    project_name="openai-agents",
    auto_instrument=True,
)

## 创建智能体

In [ ]:
from textwrap import dedent
from typing import Literal

from agents import Agent, Runner, TResponseInputItem
from pydantic import BaseModel, Field

CATALYSTS = """营收增长、利润率提升、护城河扩大、自由现金流生成、使用量、定价、分销、股票回购、股息、新产品、监管、竞争、管理团队、合并、收购、分析师评级、交易量、技术指标、价格动量"""


class EvaluationFeedback(BaseModel):
    feedback: str = Field(
        description=f"指定股票代码的正负催化因素研究报告缺少哪些内容？催化因素包括以下方面的变化： {CATALYSTS}."
    )
    score: Literal["pass", "needs_improvement", "fail"] = Field(
        description="研究报告评级。报告完整且包含正确股票代码的至少 3 个正面和 3 个负面催化因素时为 pass；缺少部分信息时为 needs_improvement；完全错误时为 fail。"
    )


report_agent = Agent(
    name="Catalyst Report Agent",
    instructions=dedent(
        """你是专注股票研究的研究助手。根据给定股票代码，生成报告，列出可能影响未来股价的 3 个正面和 3 个负面催化因素，总长不超过 50 个词。"""
    ),
    model="gpt-4.1",
)

evaluation_agent = Agent(
    name="Evaluation Agent",
    instructions=dedent(
        """你是资深金融分析师。你会收到包含正面与负面催化因素的股票研究报告。请评估报告并反馈需要改进之处。"""
    ),
    model="gpt-4.1",
    output_type=EvaluationFeedback,
)

report_feedback = "fail"
input_items: list[TResponseInputItem] = [{"content": "AAPL", "role": "user"}]

while report_feedback != "pass":
    report = await Runner.run(report_agent, input_items)
    print("### 报告 ###")
    print(report.final_output)
    input_items = report.to_input_list()

    evaluation = await Runner.run(evaluation_agent, str(report.final_output))
    evaluation_feedback = evaluation.final_output_as(EvaluationFeedback)
    print("### 评估 ###")
    print(str(evaluation_feedback))
    report_feedback = evaluation_feedback.score

    if report_feedback != "pass":
        print("根据反馈重新运行")
        input_items.append({"content": f"反馈： {evaluation_feedback.feedback}", "role": "user"})